# 11 · `gl_engine/interp/nodes.py`

## What this file is for

**ISO's entire vocabulary — 54 instructions, one function each.**

This is the file that makes the interpreter an interpreter. ISO's rules are written in a small language: if, and, or, equals, choose, look this up, multiply those, round that. Somebody had to write down what each of those 54 things *means*, exactly once. That is this file, and it was the largest single piece of specification work in the build.

The payoff is that nothing here knows anything about insurance. There is no deductible instruction and no territory instruction — there is `Lookup`, and ISO's content decides what gets looked up.

**Depends on:** [`08-interp-values`](08-interp-values.ipynb), [`09-interp-tree`](09-interp-tree.ipynb).

## Its public surface

One dispatch table, keyed by element name.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from gl_engine.interp import nodes

print("instructions:", len(nodes.EVAL))
print()
for i, name in enumerate(sorted(nodes.EVAL), 1):
    print(f"{name:<26}", end="" if i % 3 else "\n")

## The smallest thing that works

Evaluate a single instruction, by hand, outside any rating.

In [ ]:
import xml.etree.ElementTree as ET
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook
from gl_engine.interp.interpreter import Interpreter, Frame
from gl_engine.interp.program import Program
from gl_engine.interp import tree as T

NS = 'xmlns="http://www.verisk.com/iso/erc/Rule"'

book  = ResolvedBook(EditionResolver().resolve("GA", "20260811"))
ip    = Interpreter(book)
frame = Frame(data=T.Node.from_dict("GeneralLiability", {"StateCode": "GA"}),
              program=Program(book.parent.package),
              rule_file="GeneralLiabilityRules")

lit = ET.fromstring(f'<Constant {NS} Type="decimal">1.5</Constant>')
print("Constant ->", repr(ip.eval(lit, frame)))

ref = ET.fromstring(f'<Value {NS} FromDataDef="StateCode" Type="string"/>')
print("Value    ->", repr(ip.eval(ref, frame)))

Two instructions, evaluated in isolation. `Constant` carries its payload as text; `Value` reads from the data tree by name. Every one of the other 52 works the same way — a function that takes the interpreter, the element and the current frame.

## The interesting case

### The language, grouped by what it is for

Seeing them sorted alphabetically hides the shape. Grouped, the whole language fits on a page.

In [ ]:
groups = {
  "control flow"   : ["Sequence", "If", "Choose", "When", "Break", "Wrapper", "ForEach"],
  "logic"          : ["And", "Or", "Not", "Equal", "NotEqual", "GreaterThan", "LessThan",
                      "GreaterThanOrEqual", "LessThanOrEqual", "Test"],
  "arithmetic"     : ["Add", "Subtract", "Multiply", "Divide", "Round", "Min", "Max", "Sum"],
  "data"           : ["Value", "Constant", "Copy", "Count", "FirstValue", "FirstNonNull", "Param", "Arg"],
  "tables"         : ["Lookup"],
  "calling"        : ["RunRule", "Rule"],
  "text and dates" : ["Concat", "Convert", "DateAdd", "DateCreate", "DateDifference"],
}
known = {n for g in groups.values() for n in g}
for label, names in groups.items():
    present = [n for n in names if n in nodes.EVAL]
    print(f"{label:<16} {len(present):>2}  {', '.join(present)}")
print(f"\n{'ungrouped':<16} {len(set(nodes.EVAL) - known):>2}  "
      f"{', '.join(sorted(set(nodes.EVAL) - known))}")

**One instruction does the insurance.** `Lookup` reads a table. Everything else is a general-purpose programming language — which is precisely why the same 5,000 lines rate 51 jurisdictions without a single per-state branch.

### What one handler looks like

They are small, and their docstrings carry the decisions.

In [ ]:
import inspect

for fn in (nodes._constant, nodes._first_non_null):
    print("=" * 68)
    print(inspect.getsource(fn))

`FirstNonNull` is worth reading twice — it is the instruction behind *"use the state's value, or the national one if the state filed none"*. It is also where **OI-88** lives: when the value arrives through arithmetic, a null refuses instead of falling through, so the countrywide fallback never happens.

## What it refuses

An instruction the file has never heard of stops the rating.

In [ ]:
from gl_engine.interp.values import InterpretError

unknown = ET.fromstring(f'<SomethingISOInvented {NS}/>')
try:
    ip.eval(unknown, frame)
except InterpretError as e:
    print("InterpretError:", str(e).split("--")[0].strip())

print()
print("That message is a monitor: if ISO ever files a 55th instruction,")
print("the engine says so immediately instead of quietly skipping it.")

## Try it yourself

1. Build an `<Add>` of two `<Constant>` elements and evaluate it. What type comes back?
2. Find the handler for `Lookup`. How much of the work does it do itself, and how much does it hand to the interpreter?
3. Which handlers can return `None`? Compare that list against notebook 08's null discussion.

In [ ]:
# your turn